In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from io import StringIO

In [3]:
url = "https://nsearchives.nseindia.com/content/indices/ind_nifty500list.csv"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

nifty500 = pd.read_csv(StringIO(response.text))

print(nifty500.head())

               Company Name                Industry     Symbol Series  \
0          360 ONE WAM Ltd.      Financial Services     360ONE     EQ   
1            ABB India Ltd.           Capital Goods        ABB     EQ   
2                  ACC Ltd.  Construction Materials        ACC     EQ   
3  ACME Solar Holdings Ltd.                   Power  ACMESOLAR     EQ   
4      AIA Engineering Ltd.           Capital Goods     AIAENG     EQ   

      ISIN Code  
0  INE466L01038  
1  INE117A01022  
2  INE012A01025  
3  INE622W01025  
4  INE212H01026  


In [4]:
symbols = nifty500["Symbol"].dropna().tolist()

# Convert NSE symbols to Yahoo Finance format
tickers = [symbol + ".NS" for symbol in symbols]

print("Total stocks:", len(tickers))
print(tickers[:10])

Total stocks: 501
['360ONE.NS', 'ABB.NS', 'ACC.NS', 'ACMESOLAR.NS', 'AIAENG.NS', 'APLAPOLLO.NS', 'AUBANK.NS', 'AWL.NS', 'AADHARHFC.NS', 'AARTIIND.NS']


In [5]:
all_prices = []

for i in range(0, len(tickers), 50):

    batch = tickers[i:i+50]

    data = yf.download(
        tickers=batch,
        period="3y",
        interval="1d",
        auto_adjust=False,
        group_by="column",
        threads=True,
        progress=True
    )

    if not data.empty:
        adj_close = data["Adj Close"]
        all_prices.append(adj_close)

# Combine all downloaded stocks
adj_close_prices = pd.concat(all_prices, axis=1)

print(adj_close_prices.head())

[*********************100%***********************]  50 of 50 completed
[**********************96%*********************  ]  48 of 50 completed$BAGMANE.NS: No data found, symbol may be delisted
[*********************100%***********************]  50 of 50 completed

1 Failed download:
['BAGMANE.NS']: No data found, symbol may be delisted
[************          26%                       ]  13 of 50 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DUMMYHEG.NS"}}}
[**********************50%                       ]  25 of 50 completed$DUMMYHEG.NS: No data found, symbol may be delisted
[*********************100%***********************]  50 of 50 completed

1 Failed download:
['DUMMYHEG.NS']: No data found, symbol may be delisted
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************

Ticker       360ONE.NS  AADHARHFC.NS  AARTIIND.NS     AAVAS.NS       ABB.NS  \
Date                                                                          
2023-11-17  525.951477           NaN   523.548340  1444.400024  4163.794434   
2023-11-20  526.047729           NaN   513.956970  1503.349976  4223.991211   
2023-11-21  530.234192           NaN   517.932617  1525.949951  4222.338867   
2023-11-22  519.695923           NaN   517.932617  1483.650024  4114.577637   
2023-11-23  519.984558           NaN   526.530151  1488.150024  4139.987305   

Ticker      ABBOTINDIA.NS  ABCAPITAL.NS  ABDL.NS     ABREL.NS  ABSLAMC.NS  \
Date                                                                        
2023-11-17   22444.017578    170.949997      NaN  1222.461548  427.942535   
2023-11-20   22776.798828    171.149994      NaN  1221.317383  420.631622   
2023-11-21   22752.859375    171.649994      NaN  1236.840576  417.977386   
2023-11-22   22708.826172    170.949997      NaN  1226.591187

In [6]:
adj_close_prices.to_csv("nifty500_adjusted_close.csv")